# Bandit con Thompson Sampling

Siguiendo la lógica de `sesion5_03_bandits_personalizados.ipynb`, en vez de usar un solo agente para todo, aca usamos un agente por contexto. En nuestro caso, el contexto es la combinacion `audiencia + objetivo de campaña`.


## Por qué tiene sentido este enfoque

El EDA mostró algo importante: el mejor canal no es el mismo para todas las combinaciones. Justamente por eso el multi armed bandit contextual tiene sentido aca. La idea es la misma que en el notebook del curso: cada segmento necesita su propio agente para aprender mejor.


In [1]:
from pathlib import Path
from pprint import pprint
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data.carga_datos import cargar_dataset
from src.data.limpieza import limpiar_dataset_publicidad
from src.data.features import crear_contexto, agregar_metricas_por_contexto_y_canal, preparar_observaciones_bandit
from src.modelos.bandit_thompson import entrenar_agente_global, entrenar_bandits_por_contexto, recomendar_canal, comparar_bandit_contextual_vs_global


In [2]:
df = cargar_dataset(ROOT / 'social_media_ads_filtered.csv')
df = crear_contexto(limpiar_dataset_publicidad(df))
tabla_agregada = agregar_metricas_por_contexto_y_canal(df)
observaciones = preparar_observaciones_bandit(df)

agentes, politica = entrenar_bandits_por_contexto(observaciones)
agente_global = entrenar_agente_global(observaciones)
comparacion = comparar_bandit_contextual_vs_global(agentes, agente_global)

print('Numero de contextos entrenados:', len(agentes))
print('Canal aprendido con mejor rendimiento por el agente global:', agente_global.mejor_canal_aprendido())
politica.head(10)


Entrenando contexto: Women 45-60 | Brand Awareness
Entrenando contexto: Men 25-34 | Product Launch
Entrenando contexto: Men 18-24 | Brand Awareness
Entrenando contexto: Men 35-44 | Market Expansion
Entrenando contexto: Women 25-34 | Increase Sales
Entrenando contexto: Women 35-44 | Brand Awareness
Entrenando contexto: Men 25-34 | Increase Sales
Entrenando contexto: Women 45-60 | Product Launch
Entrenando contexto: Women 35-44 | Market Expansion
Entrenando contexto: Men 45-60 | Market Expansion
Entrenando contexto: Women 18-24 | Product Launch
Entrenando contexto: Women 18-24 | Brand Awareness
Entrenando contexto: Men 35-44 | Increase Sales
Entrenando contexto: Men 45-60 | Increase Sales
Entrenando contexto: Women 35-44 | Product Launch
Entrenando contexto: Women 25-34 | Brand Awareness
Entrenando contexto: Men 45-60 | Brand Awareness
Entrenando contexto: Men 35-44 | Brand Awareness
Entrenando contexto: Men 18-24 | Increase Sales
Entrenando contexto: Women 25-34 | Product Launch
Entrena

,Contexto,canal_recomendado
0,All Ages | Brand Awareness,Twitter
1,All Ages | Increase Sales,Twitter
2,All Ages | Market Expansion,Facebook
3,All Ages | Product Launch,Twitter
4,Men 18-24 | Brand Awareness,Facebook
5,Men 18-24 | Increase Sales,Instagram
6,Men 18-24 | Market Expansion,Twitter
7,Men 18-24 | Product Launch,Instagram
8,Men 25-34 | Brand Awareness,Facebook
9,Men 25-34 | Increase Sales,Instagram


## Qué aprendió un agente puntual

Aca revisamos un contexto concreto para ver cómo se distribuyó la exploración y qué canal terminó quedando arriba.


In [3]:
contexto_ejemplo = politica.iloc[0]['Contexto']
print('Contexto ejemplo:', contexto_ejemplo)
pprint(agentes[contexto_ejemplo].obtener_estadisticas())


Contexto ejemplo: All Ages | Brand Awareness
       canal  impresiones  recompensa_promedio  roi_promedio_observado  alpha  \
3    Twitter           77             0.551769                4.414156   43.0   
0   Facebook           16             0.401094                3.208750    6.0   
1  Instagram           10             0.320625                2.565000    3.0   
2  Pinterest            5             0.099786                0.798285    1.0   

   beta  muestra_esperada  
3  36.0          0.544304  
0  12.0          0.333333  
1   9.0          0.250000  
2   6.0          0.142857  


## Comparación entre agente global y agentes contextuales

En esta tabla ya no solo vemos si coinciden en el canal elegido. Tambien aparece la `recompensa_promedio`, el `roi_promedio` y la columna `mejor_agente`, así que la comparación queda bastante más útil.


In [4]:
# Vemos la comparación de rendimiento entre el agente global y los agentes contextuales
# En la mayoría de los casos, el agente contextual supera al global, pero no siempre.
# Eso refuerza la idea de que el problema sí cambia por contexto.

comparacion.head(20)


,Contexto,canal_contextual,recompensa_promedio_contextual,roi_promedio_contextual,canal_global,recompensa_promedio_global,roi_promedio_global,coinciden,mejor_agente
0,Women 45-60 | Brand Awareness,Facebook,0.579684,4.637473,Twitter,0.497152,3.977217,0,contextual
1,Men 25-34 | Product Launch,Instagram,0.507546,4.060370,Twitter,0.497152,3.977217,0,contextual
2,Men 18-24 | Brand Awareness,Facebook,0.515592,4.124737,Twitter,0.497152,3.977217,0,contextual
3,Men 35-44 | Market Expansion,Instagram,0.508710,4.069677,Twitter,0.497152,3.977217,0,contextual
4,Women 25-34 | Increase Sales,Instagram,0.562545,4.500357,Twitter,0.497152,3.977217,0,contextual
5,Women 35-44 | Brand Awareness,Twitter,0.587261,4.698088,Twitter,0.497152,3.977217,1,contextual
6,Men 25-34 | Increase Sales,Instagram,0.545147,4.361176,Twitter,0.497152,3.977217,0,contextual
7,Women 45-60 | Product Launch,Twitter,0.564162,4.513297,Twitter,0.497152,3.977217,1,contextual
8,Women 35-44 | Market Expansion,Instagram,0.568688,4.549500,Twitter,0.497152,3.977217,0,contextual
9,Men 45-60 | Market Expansion,Facebook,0.533581,4.268644,Twitter,0.497152,3.977217,0,contextual


## Ejemplos de recomendación

Aca probamos un contexto visto y un cold start total. 

Esto significa que el agente puede manejar casos de audiencias y tipos de campaña nuevos, que no se encuentran en la data. En este caso Teens y lead generation

In [5]:
ejemplo_visto = recomendar_canal(agentes, tabla_agregada, 'Women 45-60', 'Market Expansion')
ejemplo_nuevo = recomendar_canal(agentes, tabla_agregada, 'Teens 13-17', 'Lead Generation')

print('Contexto visto:')
pprint(ejemplo_visto)

print('\nCold start total:')
pprint(ejemplo_nuevo)


Contexto visto:
{'contexto': 'Women 45-60 | Market Expansion',
 'detalle': [{'alpha': 77.0,
              'beta': 24.0,
              'canal': 'Twitter',
              'impresiones': 99,
              'muestra_esperada': 0.7623762376237624,
              'recompensa_promedio': 0.619166666666667,
              'roi_promedio_observado': 4.953333333333336},
             {'alpha': 6.0,
              'beta': 6.0,
              'canal': 'Instagram',
              'impresiones': 10,
              'muestra_esperada': 0.5,
              'recompensa_promedio': 0.5077499999999999,
              'roi_promedio_observado': 4.061999999999999},
             {'alpha': 1.0,
              'beta': 4.0,
              'canal': 'Facebook',
              'impresiones': 3,
              'muestra_esperada': 0.2,
              'recompensa_promedio': 0.38833333333333336,
              'roi_promedio_observado': 3.106666666666667},
             {'alpha': 1.0,
              'beta': 4.0,
              'canal': 'Pinte

In [6]:
# Exploramos el mejor canal aprendido por agente por contexto
for contexto, agente in agentes.items():
    print(f"Contexto: {contexto} - Mejor canal aprendido: {agente.mejor_canal_aprendido()}")


Contexto: Women 45-60 | Brand Awareness - Mejor canal aprendido: Facebook
Contexto: Men 25-34 | Product Launch - Mejor canal aprendido: Instagram
Contexto: Men 18-24 | Brand Awareness - Mejor canal aprendido: Facebook
Contexto: Men 35-44 | Market Expansion - Mejor canal aprendido: Instagram
Contexto: Women 25-34 | Increase Sales - Mejor canal aprendido: Instagram
Contexto: Women 35-44 | Brand Awareness - Mejor canal aprendido: Twitter
Contexto: Men 25-34 | Increase Sales - Mejor canal aprendido: Instagram
Contexto: Women 45-60 | Product Launch - Mejor canal aprendido: Twitter
Contexto: Women 35-44 | Market Expansion - Mejor canal aprendido: Instagram
Contexto: Men 45-60 | Market Expansion - Mejor canal aprendido: Facebook
Contexto: Women 18-24 | Product Launch - Mejor canal aprendido: Instagram
Contexto: Women 18-24 | Brand Awareness - Mejor canal aprendido: Twitter
Contexto: Men 35-44 | Increase Sales - Mejor canal aprendido: Twitter
Contexto: Men 45-60 | Increase Sales - Mejor canal 

## Análisis

Acá ya se ve bien la conexión con el notebook del curso: un solo agente global simplifica demasiado el problema, mientras que un agente por contexto tiene más sentido porque cada combinación aprende su propio patrón. En publicidad eso es clave, porque la audiencia cambia, el objetivo cambia y el canal que conviene también puede cambiar.
